Created 8/20/2026

Purpose: To mess around and test out the cleaned up code and functionality created by claude.

In [ ]:
import numpy as np
import pandas as pd
import os
import sys

from tess_window import aliases, experiment, mission, plots, targets, transits
from tess_window.aliases import AMBIGUOUS, ERROR, MONO, NO_TRANSIT, SOLVED, AliasResult

from tess_window import aliases as al
from tess_window.mission import mission_span
from tess_window.transits import (check_observability, resolve_duration,
                       transit_times_from_phase)
from tess_window.targets import

In [ ]:
periods = np.arange(20,30)
tc_phases = np.arange(0, 1, .1)

run_grid(periods, tc_phases, sector_sets, windows, duration=0.0,
             min_period=13.0, start_time=None, stop_time=None, progress=False)

In [ ]:
## claude copying starts from here

In [ ]:
# setup and sanity check
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from astropy.time import Time

from tess_window import mission, targets, transits, experiment, plots
from tess_window import aliases as al

windows = mission.load_orbit_windows()          # errata applied, auto cutoff
report  = mission.validate_sector_alignment(windows, verbose=True)

start, stop = mission.mission_span(windows)
print(f"sectors 1-{windows['sector'].max()}, {len(windows)} windows")
print(f"{Time(start,format='jd').iso[:10]} -> {Time(stop,format='jd').iso[:10]}"
      f"  ({stop-start:.0f} d)")

In [ ]:
# check the errata took effect
b = mission.sector_bounds(windows)
b["span"] = b["t_end"] - b["t_start"]
b.tail(8)[["n_windows", "span"]].round(2)

In [ ]:
# sky positions
tg   = targets.load_targets(max_sector=int(windows["sector"].max()))
sets = targets.unique_sector_sets(tg, min_sectors=2)
print(f"{len(tg):,} targets -> {len(sets):,} distinct sector combinations")
sets.sort_values("weight", ascending=False).head()

In [ ]:
# inject one planet and trace it
target = sets.sort_values("weight", ascending=False).iloc[0]
period = 137.0
dur    = transits.transit_duration(period)

# at long period most phases miss entirely; scan for one that lands 2+ transits
for phase in np.linspace(0, 1, 200, endpoint=False):
    tt = transits.transit_times_from_phase(period, phase, start, stop)
    obs, win_obs, visible = transits.check_observability(
        tt, target["sectors"], windows, duration=dur)
    if obs[0].sum() >= 2:
        break

print(f"sectors {target['sectors']}  ec_lat={target['ec_lat']:.1f} deg")
print(f"P={period} d, dur={24*dur:.1f} hr, phase={phase:.3f}")
print(f"{obs[0].sum()} of {tt.shape[1]} transits observed")
print(f"{len(visible)} visible windows, {win_obs[0].sum()} caught a transit")

In [ ]:
# alias analysis
res = al.analyze(tt[0][obs[0]], visible[~win_obs[0]], duration=dur)
print("flag:", al.FLAG_NAMES[res.flag])
print(f"{len(res.alias_periods)} candidate aliases, {res.n_surviving} survive")
print(np.round(res.surviving, 3))

# plots
plots.plot_transits(tt[0], obs[0], visible, win_obs[0],
                    title=f"P = {period} d")
plots.plot_aliases(res, visible, win_obs[0], stop);

In [ ]:
# mini batch
mini    = sets.sample(40, weights=sets["weight"], random_state=1)
periods = np.geomspace(20, 400, 12)
phases  = np.linspace(0, 1, 6, endpoint=False)

results = experiment.run_grid(periods, phases, mini, windows,
                              duration=transits.transit_duration, progress=True)
print(f"{len(results):,} systems")
results["flag"].map(al.FLAG_NAMES).value_counts()

In [ ]:
# completion curve
summary = experiment.summarize(results)      # star-weighted
plots.plot_completion(summary);
summary[["period","solved","ambiguous","mono","missed"]].round(3).head()

In [ ]:
# score a future strategy
pointings = np.arange(1, 14)
slots = np.arange(last + 1, last + 1 + len(pointings))
future = mission.synthesize_windows(windows, slots, pointings)

scored = experiment.score_strategy(results, windows, future, mini)

print(experiment.transition_matrix(scored, weighted=False))
print(scored.loc[scored["upgraded"], "transition"].value_counts())

before = experiment.summarize(results)
after = experiment.summarize(scored, flag_col="flag_new")

In [ ]:
# the correctness invariant
rng, bad = np.random.default_rng(0), 0
for _ in range(200):
    p = rng.uniform(15, 400)
    t = mini.iloc[rng.integers(len(mini))]
    d = transits.transit_duration(p)
    tt_ = transits.transit_times_from_phase(p, rng.random(), start, stop)
    o, w_, vis = transits.check_observability(tt_, t["sectors"], windows, duration=d)
    r = al.analyze(tt_[0][o[0]], vis[~w_[0]], duration=d)
    if r.flag in (al.NO_TRANSIT, al.MONO):
        continue
    bad += np.min(np.abs(r.surviving - p)) > 0.1
print("true period ruled out in", bad, "cases (must be 0)")